# 03 · Casting Product - Transfer Learning + MLflow

Training ResNet18 for binary classification of casting defects.
All experiments are logged to MLflow (`SQLite (mlflow.db in project root)`).

In [ ]:
import sys
from pathlib import Path

# Add notebooks/utils to path (works on Mac, Windows, Linux)
_nb_root = Path('../..').resolve()
if str(_nb_root) not in sys.path:
    sys.path.insert(0, str(_nb_root))

from utils.arkon_utils import (
    get_device, get_mlflow_uri, save_figure,
    Timer, CheckpointManager, recommended_num_workers
)

print('arkon_utils loaded ✓')

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import mlflow
import mlflow.pytorch

# Detect best device (RTX 3070 on Windows → CUDA)
device = get_device()

ASSETS = 'cv/casting'


In [ ]:
# MLflow - platform-safe SQLite (no server needed)
mlflow.set_tracking_uri(get_mlflow_uri())
EXPERIMENT_NAME = 'arkon-cv-casting'
RUN_NAME        = 'resnet18_v1'
mlflow.set_experiment(EXPERIMENT_NAME)
print(f'MLflow URI: {mlflow.get_tracking_uri()}')

MLFLOW_TAGS = {
    'dataset':    'casting_product',
    'task':       'binary_classification',
    'model_arch': 'resnet18_pretrained',
    'device':     str(device),
}


In [ ]:
DATA_DIR  = Path('../../../data/03_casting/raw')
MODEL_DIR = Path('../../../models/03_casting')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR  = Path('../../../models/checkpoints/casting')

PARAMS = {
    'img_size'    : 224,
    'batch_size'  : 64 if device.type == 'cuda' else 32,  # larger batch on GPU
    'epochs'      : 10,
    'lr'          : 1e-4,
    'dropout'     : 0.3,
    'num_classes' : 2,
    'num_workers' : recommended_num_workers(device),
    'pin_memory'  : device.type == 'cuda',
    'seed'        : 42,
}
torch.manual_seed(PARAMS['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed(PARAMS['seed'])

print(f'Config: {PARAMS}')


## 1. DataLoaders

In [ ]:
MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]
SZ   = PARAMS["img_size"]

train_tf = transforms.Compose([
    transforms.Resize((SZ, SZ)), transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(), transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
val_tf = transforms.Compose([
    transforms.Resize((SZ, SZ)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

full_train = datasets.ImageFolder(DATA_DIR / 'train', transform=train_tf)
test_set   = datasets.ImageFolder(DATA_DIR / 'test',  transform=val_tf)
val_size   = int(PARAMS["val_split"] * len(full_train))
train_set, val_set = random_split(full_train, [len(full_train)-val_size, val_size],
                                  generator=torch.Generator().manual_seed(PARAMS["seed"]))
val_set.dataset.transform = val_tf
CLASSES = full_train.classes

BS = PARAMS["batch_size"]
train_loader = DataLoader(train_set, BS, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_set,   BS, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_set,  BS, shuffle=False, num_workers=0)
print(f"Classes: {CLASSES} | Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

## 2. Model

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Sequential(nn.Dropout(PARAMS["dropout"]),
                         nn.Linear(512, PARAMS["num_classes"]))
model = model.to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=PARAMS["lr"])
scheduler = StepLR(optimizer, step_size=PARAMS["scheduler_step"],
                   gamma=PARAMS["scheduler_gamma"])
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 3. Training + MLflow Run

In [ ]:
# scaler enables Mixed Precision on CUDA; on CPU/MPS it's a no-op
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    loss_sum, correct, total = 0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        with torch.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            out  = model(X)
            loss = criterion(out, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.item() * len(y)
        correct  += (out.argmax(1) == y).sum().item()
        total    += len(y)
    return loss_sum / total, correct / total

def eval_epoch(model, loader, criterion, device):
    model.eval()
    loss_sum, correct, total = 0, 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            with torch.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                out  = model(X)
                loss = criterion(out, y)
            loss_sum += loss.item() * len(y)
            correct  += (out.argmax(1) == y).sum().item()
            total    += len(y)
    return loss_sum / total, correct / total


In [ ]:
ckpt = CheckpointManager(CKPT_DIR)

if ckpt.exists_torch('resnet18_v1_best'):
    print('\n⚡ Checkpoint found - skipping training, loading best model...')
    state, meta = ckpt.load_torch('resnet18_v1_best', map_location=device)
    model.load_state_dict(state)
    history = meta.get('history', {})
    train_time_str = meta.get('train_time', 'unknown')
    print(f'  Best epoch    : {meta.get("best_epoch", "?")}')
    print(f'  Val accuracy  : {meta.get("val_acc", "?")}')
    print(f'  Training time : {train_time_str}')
else:
    with mlflow.start_run(run_name=RUN_NAME, tags=MLFLOW_TAGS) as run:
        mlflow.log_params(PARAMS)
        print(f'Run ID: {run.info.run_id}')

        history = {k: [] for k in ['train_loss','val_loss','train_acc','val_acc']}
        best_val_acc = 0.0
        best_epoch   = 0

        with Timer('ResNet18 training') as train_timer:
            for epoch in range(1, PARAMS['epochs'] + 1):
                tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, device)
                va_loss, va_acc = eval_epoch(model, val_loader,   criterion, device)
                scheduler.step()

                history['train_loss'].append(tr_loss)
                history['val_loss'].append(va_loss)
                history['train_acc'].append(tr_acc)
                history['val_acc'].append(va_acc)

                mlflow.log_metrics({'train_loss': tr_loss, 'val_loss': va_loss,
                                    'train_acc': tr_acc,  'val_acc':  va_acc}, step=epoch)

                print(f'Epoch {epoch:02d}/{PARAMS["epochs"]}  '
                      f'train_loss={tr_loss:.4f}  val_loss={va_loss:.4f}  '
                      f'train_acc={tr_acc:.4f}  val_acc={va_acc:.4f}')

                if va_acc > best_val_acc:
                    best_val_acc = va_acc
                    best_epoch   = epoch
                    ckpt.save_torch(model.state_dict(), 'resnet18_v1_best',
                                    metadata={'epoch': epoch, 'val_acc': va_acc,
                                              'history': history})

                # Save epoch checkpoint for resuming if interrupted
                ckpt.save_epoch(model, optimizer, epoch,
                                {'val_acc': va_acc, 'val_loss': va_loss}, 'resnet18_v1')

        train_time_str = train_timer.report()
        mlflow.log_param('train_time', train_time_str)
        mlflow.log_param('best_epoch', best_epoch)
        mlflow.log_metric('best_val_acc', best_val_acc)

        # Update best checkpoint with final metadata
        state, _ = ckpt.load_torch('resnet18_v1_best', map_location=device)
        model.load_state_dict(state)
        ckpt.save_torch(model.state_dict(), 'resnet18_v1_best',
                        metadata={'best_epoch': best_epoch, 'val_acc': best_val_acc,
                                  'train_time': train_time_str, 'history': history})

        print(f'\n✓ Training complete | Best val_acc={best_val_acc:.4f} @ epoch {best_epoch}')
        print(f'⏱ {train_time_str}')


## Summary

| Parameter | Value |
|---|---|
| MLflow Experiment | `arkon-cv-casting` |
| Model Registry | `casting_resnet18` |
| Logged | params, metrics/epoch, curves, confusion matrix, weights |
| Server | `http://SQLite (mlflow.db in project root)` |

In [ ]:
# ── Training curves ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_range, history['train_loss'], label='Train')
axes[0].plot(epochs_range, history['val_loss'],   label='Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(epochs_range, history['train_acc'], label='Train')
axes[1].plot(epochs_range, history['val_acc'],   label='Val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()

plt.suptitle(f'ResNet18 - Casting Defect Detection  |  train time: {train_time_str}')
plt.tight_layout()
save_figure(fig, 'casting_model_training_curves', subfolder=ASSETS)
plt.show()
